## Why are the embedding tables empty?

`ticker_news_embeddings` and `ticker_news_chunk_embeddings` have **zero lifetime
inserts** — nothing has ever been written to them, so the ingest notebook failed
before reaching the database.

This notebook isolates the cause. Each test checks one link in the chain and prints
`PASS` / `FAIL` / `SKIP`. The last cell prints a verdict naming the first broken link.

Tests are ordered cheapest-first: connection and permissions before the model
download, so a failure surfaces in seconds rather than after a multi-GB install.

**Run on a classic cluster.** Nothing here writes permanent data — the one write
test inserts a probe row and deletes it.

In [ ]:
# Records results so the final cell can name the first failure.
RESULTS = []

def record(name, ok, detail="", skipped=False):
    status = "SKIP" if skipped else ("PASS" if ok else "FAIL")
    RESULTS.append({"test": name, "status": status, "detail": str(detail)[:300]})
    print(f"[{status}] {name}")
    if detail:
        for line in str(detail).splitlines():
            print(f"       {line}")
    return ok

print("initialised")

In [ ]:
# dbutils.secrets.get() returns the DECODED value (unlike the SDK's
# get_secret(), which returns base64). psycopg2 accepts the URL directly.
from urllib.parse import urlparse, unquote

LAKEBASE_URL = None
try:
    LAKEBASE_URL = dbutils.secrets.get(scope="database", key="lakebase-url")
    p = urlparse(LAKEBASE_URL)
    detail = (f"host={p.hostname}\n"
              f"database={p.path.lstrip('/')}  user={p.username}\n"
              f"password={len(p.password or '')} chars  sslmode={'require' in (p.query or '')}")
    ok = bool(p.hostname and p.username and p.password)
    if p.hostname and "neon.tech" in p.hostname:
        detail += "\nWARNING: a .neon.tech host is the old/incorrect endpoint"
        ok = False
    record("1. Secret readable and URL well-formed", ok, detail)
except Exception as e:
    record("1. Secret readable and URL well-formed", False, e)

In [ ]:
import psycopg2

conn = None
try:
    conn = psycopg2.connect(LAKEBASE_URL)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("SELECT current_user, current_database(), version()")
    usr, db, ver = cur.fetchone()
    record("2. Connect to Lakebase", True,
           f"user={usr}  database={db}\n{ver.split(' on ')[0]}")
except Exception as e:
    record("2. Connect to Lakebase", False, e)

In [ ]:
TABLES = ["watchlist", "ticker_news_documents",
          "ticker_news_embeddings", "ticker_news_chunk_embeddings"]
try:
    missing, counts = [], []
    for t in TABLES:
        cur.execute("SELECT to_regclass(%s) IS NOT NULL", (f"public.{t}",))
        if not cur.fetchone()[0]:
            missing.append(t)
            continue
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        counts.append(f"{t}: {cur.fetchone()[0]} rows")
    record("3. All four tables exist", not missing,
           "\n".join(counts) + (f"\nMISSING: {missing}" if missing else ""))
except Exception as e:
    record("3. All four tables exist", False, e)

In [ ]:
# atttypmod holds the declared width of a VECTOR(n) column. A mismatch against
# the model's output dimension makes every insert fail.
EXPECTED_DIM = 384   # sentence-transformers/all-MiniLM-L6-v2
try:
    lines, ok = [], True
    for t in ["ticker_news_embeddings", "ticker_news_chunk_embeddings"]:
        cur.execute(f"""SELECT atttypmod FROM pg_attribute
                        WHERE attrelid='{t}'::regclass AND attname='embedding'""")
        dim = cur.fetchone()[0]
        lines.append(f"{t}.embedding = VECTOR({dim})")
        ok = ok and dim == EXPECTED_DIM
    record(f"4. Vector columns are VECTOR({EXPECTED_DIM})", ok, "\n".join(lines))
except Exception as e:
    record(f"4. Vector columns are VECTOR({EXPECTED_DIM})", False, e)

In [ ]:
# n_tup_ins is a LIFETIME counter - it counts every row ever inserted, including
# rows since deleted, and is not reset by DELETE. So it reveals whether a table
# has ever been written to, even when it currently reads 0 rows.
#
# Caveat: probe rows from diagnostics (test 6 below, and any earlier debugging)
# also land in this counter. Compare against the document count instead of
# against zero: a real pipeline run embeds roughly one row per document, so a
# handful of inserts means probes, not a completed run.
try:
    cur.execute("SELECT COUNT(*) FROM ticker_news_documents")
    doc_count = cur.fetchone()[0]
    cur.execute("""SELECT relname, n_tup_ins, n_tup_upd, n_live_tup
                   FROM pg_stat_user_tables ORDER BY relname""")
    rows = cur.fetchall()
    lines = [f"{r[0]:<32} lifetime_inserts={r[1]:<6} updates={r[2]:<6} live={r[3]}"
             for r in rows]
    ever = {r[0]: r[1] for r in rows}

    embedded_ever = ever.get("ticker_news_embeddings", 0)
    ran = embedded_ever >= max(doc_count, 1)
    verdict = (f"documents: {doc_count}; embeddings ever inserted: {embedded_ever}\n"
               + ("-> consistent with a completed pipeline run" if ran else
                  "-> far fewer inserts than documents: the pipeline never completed."
                  "\n   (a small nonzero count is diagnostic probe rows, not real output)"))
    record("5. Pipeline has actually written embeddings", ran,
           "\n".join(lines) + "\n" + verdict)
except Exception as e:
    record("5. Pipeline has actually written embeddings", False, e)

In [ ]:
# Proves the role may INSERT a real vector. Probe row is removed immediately.
try:
    literal = "[" + ",".join(["0.01"] * EXPECTED_DIM) + "]"
    cur.execute("""INSERT INTO ticker_news_embeddings
                     (id, ticker, title, embedding, model_name, embedded_at)
                   VALUES ('_diag_probe','TEST','probe', %s::vector,'_diag', now())
                   ON CONFLICT (id) DO NOTHING""", (literal,))
    cur.execute("""SELECT pg_typeof(embedding)::text, vector_dims(embedding)
                   FROM ticker_news_embeddings WHERE id='_diag_probe'""")
    got = cur.fetchone()
    cur.execute("DELETE FROM ticker_news_embeddings WHERE id='_diag_probe'")
    record("6. Can INSERT a vector (write permission + type)", bool(got),
           f"stored type={got[0]}, dims={got[1]}, probe row deleted" if got
           else "insert produced no row")
except Exception as e:
    try:
        cur.execute("DELETE FROM ticker_news_embeddings WHERE id='_diag_probe'")
    except Exception:
        pass
    record("6. Can INSERT a vector (write permission + type)", False, e)

In [ ]:
try:
    cur.execute("SELECT DISTINCT symbol FROM watchlist ORDER BY symbol")
    tickers = [r[0] for r in cur.fetchall()]
    cur.execute("""SELECT COUNT(*) FROM ticker_news_documents d
                   LEFT JOIN ticker_news_embeddings e ON e.id = d.id
                   WHERE e.id IS NULL""")
    pending = cur.fetchone()[0]
    record("7. There is work to do", bool(tickers) and pending > 0,
           f"watchlist tickers: {tickers or '(empty)'}\n"
           f"documents awaiting embeddings: {pending}")
except Exception as e:
    record("7. There is work to do", False, e)

In [ ]:
# The most likely failure: the original notebook ran
#   %pip uninstall -y psycopg2 psycopg2-binary
# then imported psycopg2, which only works if the runtime ships its own copy.
import importlib

deps = {}
for mod in ["psycopg2", "sentence_transformers", "trafilatura", "torch", "pandas"]:
    try:
        m = importlib.import_module(mod)
        deps[mod] = getattr(m, "__version__", "installed")
    except Exception as e:
        deps[mod] = f"MISSING ({type(e).__name__})"

ml_ready = not any(str(deps[m]).startswith("MISSING")
                   for m in ["sentence_transformers", "trafilatura", "torch"])
record("8. Python dependencies importable", ml_ready,
       "\n".join(f"{k:<22} {v}" for k, v in deps.items()) +
       ("" if ml_ready else
        "\n-> run: %pip install sentence-transformers trafilatura psycopg2-binary"
        "\n   then: dbutils.library.restartPython()"))

In [ ]:
# Downloads the model on first run; can take a few minutes.
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embed = None
try:
    if not ml_ready:
        record("9. Embedding model loads and returns correct dims", False,
               "skipped - dependencies missing (see test 8)", skipped=True)
    else:
        import os
        os.environ.setdefault("HF_HOME", "/tmp/.cache/huggingface")
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(MODEL_NAME)
        embed = lambda texts: model.encode(list(texts), show_progress_bar=False,
                                           convert_to_numpy=True).tolist()
        v = embed(["dimension probe"])[0]
        record("9. Embedding model loads and returns correct dims",
               len(v) == EXPECTED_DIM,
               f"{MODEL_NAME} -> {len(v)} dims (column expects {EXPECTED_DIM})")
except Exception as e:
    record("9. Embedding model loads and returns correct dims", False, e)

In [ ]:
# One request, so the free tier's 5/min limit is not a factor.
try:
    import base64, requests
    key = dbutils.secrets.get(scope="massive", key="api-key")
    r = requests.get("https://api.massive.com/v2/aggs/ticker/AAPL/prev",
                     headers={"Authorization": f"Bearer {key}"}, timeout=30)
    ok = r.status_code == 200
    record("10. Massive API key works", ok,
           f"HTTP {r.status_code}\n{str(r.text)[:200]}")
except Exception as e:
    record("10. Massive API key works", False, e)

In [ ]:
# Article bodies come from publisher sites, not Databricks - clusters in a
# locked-down VPC often cannot reach them. Failure here only disables chunk
# embeddings (stage 4); title/description embeddings still work.
try:
    if str(deps.get("trafilatura", "")).startswith("MISSING"):
        record("11. Can fetch an article body (chunk embeddings only)", False,
               "skipped - trafilatura missing", skipped=True)
    else:
        import trafilatura
        cur.execute("""SELECT article_url FROM ticker_news_documents
                       WHERE article_url IS NOT NULL LIMIT 1""")
        row = cur.fetchone()
        url = row[0] if row else None
        downloaded = trafilatura.fetch_url(url) if url else None
        text = trafilatura.extract(downloaded) if downloaded else None
        record("11. Can fetch an article body (chunk embeddings only)", bool(text),
               f"url={url}\nextracted {len(text) if text else 0} chars"
               + ("" if text else "\n-> set skip_chunks=true and still get stage-3 embeddings"))
except Exception as e:
    record("11. Can fetch an article body (chunk embeddings only)", False, e)

In [ ]:
# The whole chain on ONE document: read -> embed -> insert -> verify -> delete.
# If this passes, the pipeline works and the original notebook failed for a
# reason unrelated to the database.
try:
    if embed is None:
        record("12. End-to-end on one document", False,
               "skipped - no working embedder (see test 9)", skipped=True)
    else:
        cur.execute("""SELECT d.id, d.ticker, d.title, d.description
                       FROM ticker_news_documents d
                       LEFT JOIN ticker_news_embeddings e ON e.id = d.id
                       WHERE e.id IS NULL LIMIT 1""")
        doc = cur.fetchone()
        if not doc:
            record("12. End-to-end on one document", False,
                   "no unembedded documents available", skipped=True)
        else:
            doc_id, ticker, title, desc = doc
            vec = embed([" ".join(filter(None, [title, desc]))])[0]
            literal = "[" + ",".join(f"{float(x):.7g}" for x in vec) + "]"
            cur.execute("""INSERT INTO ticker_news_embeddings
                             (id, ticker, title, embedding, model_name, embedded_at)
                           VALUES (%s,%s,%s,%s::vector,%s, now())
                           ON CONFLICT (id) DO NOTHING""",
                        (doc_id, ticker, title, literal, MODEL_NAME))
            cur.execute("""SELECT vector_dims(embedding) FROM ticker_news_embeddings
                           WHERE id=%s""", (doc_id,))
            got = cur.fetchone()
            record("12. End-to-end on one document", bool(got),
                   f"embedded and stored '{str(title)[:50]}...' "
                   f"({got[0] if got else '?'} dims)\n"
                   f"KEPT this row - it is real data, not a probe")
except Exception as e:
    record("12. End-to-end on one document", False, e)

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS)
display(df)

failed = [r for r in RESULTS if r["status"] == "FAIL"]
print("\n" + "=" * 68)
if not failed:
    print("ALL TESTS PASSED.")
    print("The pipeline works end to end. Run notebooks/lakebase_embeddings.py")
    print("to embed the remaining documents.")
else:
    first = failed[0]
    print(f"FIRST FAILURE: {first['test']}")
    print(f"  {first['detail'][:400]}")
    print("\nFix this one first - later failures are often consequences of it.")
    if len(failed) > 1:
        print(f"\nAlso failing: {[f['test'] for f in failed[1:]]}")
print("=" * 68)

try:
    conn.close()
    print("\nconnection closed")
except Exception:
    pass